# 04 — Antibaryon veto, and cumulative Stage 1–4 performance

**Stage 4 checkpoint notebook** for the BNV searches
$B^0 \to \Lambda^0 \Lambda^0$ and $B^+ \to \Lambda_c^+ \Lambda^0$.

**Physics.** The BNV signal final state is all-baryon — two protons, or two
antiprotons for the charge-conjugate $B$ — and contains *no* antibaryon,
whereas SM decays that produce a baryon also produce a compensating
antibaryon. So a $B$ candidate is vetoed if the event holds an identified
antiproton that is not one of that candidate's own tracks. This follows the
$B^+ \to p \Lambda^0$ reference (`BNV_pLambda`'s `build_antiproton_antimask`),
generalized from its one-$B$-candidate-per-event assumption to a per-candidate
mask.

**Charge conjugation** is never hardcoded: the signal baryon charge is read
off each candidate's own $\Lambda^0$ proton daughter
($\mathrm{sign}(\texttt{Lambda0d1Lund})$), and tracks of the *opposite* charge
fire the veto — so $B$ and $\bar B$ candidates flip automatically.

Sections 1–4 use **MC only**. Section 5 (cumulative performance) additionally
reads the **BLINDED** collision file and counts data in the fit region with
the signal box **explicitly removed** — the signal box is never read.

Recommendations only: nothing here modifies `channel_config.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

%load_ext autoreload
%autoreload 2

import numpy as np
import awkward as ak
import pandas as pd
import matplotlib.pylab as plt

from channel_config import get_channel_config, BACKGROUND_SP_MODES, SIGNAL_SP_MODE
import datasets
import cutflow
import pid_selector
import pid_optimization as po
import plotting
import results_io
import cumulative_performance as cp
import run_stage04 as rs4

In [ ]:
# Select the channel here: 'Lam0Lam0' or 'Lam0LamC'
CHANNEL = 'Lam0LamC'

config = get_channel_config(CHANNEL)
fig_dir = plotting.plot_dir(config)
print(f"Channel: {config['name']}   {config['decay_label']}")

## 1. Load MC and build the pre-veto baseline

The veto's Punzi FOM is measured **on top of** the selection already in
force — Stage 2 purity **and** the Stage 3 PID cuts (both now applied in
`channel_config.py`) — so it reflects the veto's own marginal effect, the
same convention Stage 3 used when it baselined on Stage 2.

In [ ]:
data_sp, _ = datasets.load_datasets(CHANNEL, sp_or_data='sp')
datasets.add_derived_fields(data_sp, config)

weights = datasets.get_scaling_weights(BACKGROUND_SP_MODES)

baseline_mask = (cutflow.get_composite_purity_masks_per_B(data_sp, config) &
                 cutflow.get_pid_mask_per_B(data_sp, config))

print(f"B candidates: {int(ak.sum(ak.num(data_sp['BpostFitMes'])))}")
print(f"  passing Stage 2 purity + Stage 3 PID: {int(ak.sum(baseline_mask))}")

## 2. Validate the exclusion set before anything depends on it

"Not part of the signal candidate" is determined by walking each $B$
candidate's daughters down to TRK indices. For `Lam0LamC` that walk is
$\Lambda_c^+$-decay-mode dependent, so it is *checked*, not assumed: the
number of distinct resolvable tracks per candidate must equal the
final-state size.

**Known limitation (`Lam0LamC` modes 2 and 3).** The $K_S^0$'s two pion
tracks cannot be resolved from these parquet files — `K_Sd1Idx`/`K_Sd2Idx`
do not index any collection present in the file (they exceed `npi` for 94%
of $K_S^0$ candidates, and in TRK space the implied daughter charges are
opposite only 55% of the time, vs. 50% for random). Those two pions
therefore stay eligible to fire the veto, which costs a little signal
efficiency in modes 2 and 3 only. Section 4 measures that cost; modes 1 and
4 have no $K_S^0$ and are the clean comparison. This is why the expected
counts below are 5/**3**/**5**/7 rather than 5/5/7/7.

In [ ]:
track_walk = rs4.validate_track_walk(data_sp, config)

n_tracks = cutflow.count_signal_b_tracks(data_sp, config)
if CHANNEL == 'Lam0LamC':
    mode_per_b = cutflow.get_lambdac_decay_mode_per_B(data_sp, config)
    for m in sorted(config['lambdac_modes']):
        sel = (mode_per_b == m)
        vals, counts = np.unique(ak.to_numpy(ak.flatten(n_tracks[sel])), return_counts=True)
        print(f"  mode {m}: distinct resolvable tracks/B = {dict(zip(vals.tolist(), counts.tolist()))}")
else:
    vals, counts = np.unique(ak.to_numpy(ak.flatten(n_tracks)), return_counts=True)
    print(f"  distinct tracks/B = {dict(zip(vals.tolist(), counts.tolist()))}")

## 3. Punzi scan over the KM proton ladder

Same figure of merit, same $a=4$, and the same boundary/low-stats flagging as
Stage 3 — mechanically the veto is one more KM-ladder cut expressed as a
per-$B$ mask, so it reuses `pid_optimization.evaluate_combo` unchanged.

**The ladder runs the opposite way from Stage 3.** A *looser* proton selector
tags more tracks as antiprotons, so the veto fires *more* often: more
background rejection **and** more signal loss. Tightening the selector moves
toward "no veto at all".

In [ ]:
ladder = pid_selector.KM_LADDER['p']

df_veto = po.scan_antibaryon_veto(data_sp, config, weights, baseline_mask, ladder)
best_veto = po.best_from_ladder_scan(df_veto, ['p_selector_idx'])
recommended = best_veto['best_row']['p_selector']

display(df_veto[['p_selector', 'sig_eff', 'bkg_weighted', 'bkg_raw_mc_candidates', 'fom']])
print(f"\nRecommended: {recommended}  "
      f"(FOM {best_veto['best_row']['fom']:.4f}, "
      f"sig_eff {best_veto['best_row']['sig_eff']:.4f}, "
      f"bkg {best_veto['best_row']['bkg_weighted']:.2f})")
print(f"boundary-hugging: {best_veto['any_at_scan_boundary']}   "
      f"low MC stats: {best_veto['low_mc_stats_at_best']}")

In [ ]:
plotting.plot_pid_ladder_scan_1d(df_veto, 'p_selector', 'p_selector_idx',
                                 chosen_selector=recommended,
                                 title=f"{config['decay_label']} — antibaryon veto Punzi scan")
plt.savefig(f'{fig_dir}/antibaryon_veto_scan.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Comparisons the checkpoint needs

Four things are recorded so the recommendation can be judged rather than
taken on faith:

1. **No veto at all** — is the veto worth applying? (mirrors Stage 3's
   no-PID baseline)
2. **The $p\Lambda^0$ reference choice**, `TightKMProtonSelection`, whatever
   the scan prefers.
3. **Alternative definitions** — the reference-faithful exclusion scope
   (only proton daughters exempt) and the `p`-list-only eligible-track pool.
4. **Per-$\Lambda_c^+$-mode efficiency**, which turns the unresolvable-$K_S^0$
   caveat into a number.

In [ ]:
no_veto = po.evaluate_combo(data_sp, config, weights, baseline_mask,
                            ak.ones_like(baseline_mask, dtype=bool))
ref_row = df_veto[df_veto['p_selector'] == rs4.REFERENCE_SELECTOR].iloc[0]

print(f"no veto at all           : FOM {no_veto['fom']:.4f}  bkg {no_veto['bkg_weighted']:.2f}")
print(f"recommended {recommended:<28}: FOM {best_veto['best_row']['fom']:.4f}")
print(f"p-Lambda0 ref {rs4.REFERENCE_SELECTOR:<26}: FOM {ref_row['fom']:.4f}")
print(f"\nrecommended beats no veto: "
      f"{best_veto['best_row']['fom'] > no_veto['fom']}")

In [ ]:
# Alternative definitions, at the recommended selector.
#
# 'p_list' is DEGENERATE and is recorded as the justification for using the
# per-track pSelectorsMap, not as a competing option: the 'p' hypothesis
# collection holds only ~2 candidates/event (essentially the signal protons
# themselves, all of signal-baryon charge), so it contains almost no
# opposite-charge track and the veto approaches never firing.
for label, kw in (('scope=proton_daughters_only (p-Lambda0 reference)', {'scope': 'proton_daughters_only'}),
                  ('pool=p_list (DEGENERATE)', {'pool': 'p_list'})):
    m = cutflow.get_antibaryon_veto_mask(data_sp, config, selector=recommended, **kw)
    r = po.evaluate_combo(data_sp, config, weights, baseline_mask, m)
    print(f"{label:<52}: sig_eff {r['sig_eff']:.6f}  FOM {r['fom']:.6f}")

m_default = cutflow.get_antibaryon_veto_mask(data_sp, config, selector=recommended)
r0 = po.evaluate_combo(data_sp, config, weights, baseline_mask, m_default)
print(f"{'default (all_signal_tracks / all_tracks)':<52}: "
      f"sig_eff {r0['sig_eff']:.6f}  FOM {r0['fom']:.6f}")

# Optional anti-Lambda0 add-on: measured, not adopted.
anti_lam = cutflow.get_anti_lambda0_veto_mask(data_sp, config)
r_al = po.evaluate_combo(data_sp, config, weights, baseline_mask, m_default & anti_lam)
print(f"\n+ anti-Lambda0 add-on: sig_eff {r_al['sig_eff']:.6f}  FOM {r_al['fom']:.6f}")

In [ ]:
if CHANNEL == 'Lam0LamC':
    per_mode = po.veto_efficiency_by_lambdac_mode(data_sp, config, baseline_mask, recommended)
    rows = [{'mode': m, 'label': config['lambdac_modes'][m],
             'n_before': v['n_before'], 'n_after': v['n_after'],
             'veto_efficiency': v['efficiency'],
             'K_S0 daughters unresolved': v['k0s_daughters_unresolved']}
            for m, v in per_mode.items()]
    display(pd.DataFrame(rows))
    print("Modes 1 and 4 have no K_S0 and are the clean comparison; any deficit\n"
          "in modes 2/3 is the cost of leaving the K_S0 pions eligible.")

## 5. Before/after diagnostic plots

Independent of the FOM, in the style established at the Stage 3 checkpoint
(`plotting.plot_mass_cut_diagnostic`, peak window and contiguous sidebands
shaded): the distribution before vs. after the veto, for signal MC and
luminosity-weighted background MC side by side.

For this cut the relevant peaks are the $B$'s own $m_{ES}$ and $\Delta E$ —
that is where the veto should bite into the sidebands harder than the peak.
The $\Lambda^0$ mass is shown too as a **null check**: the veto acts on
rest-of-event tracks, so it should *not* sculpt the $\Lambda^0$ peak.

In [ ]:
def veto_before_after(var, window, hist_key, title):
    """Before/after the veto for one variable, signal MC and weighted bkg MC."""
    keep = cutflow.get_antibaryon_veto_mask(data_sp, config, selector=recommended)
    spmode = data_sp['spmode']

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # --- signal MC ---
    sel = baseline_mask & (spmode == SIGNAL_SP_MODE)
    before = ak.to_numpy(ak.flatten(data_sp[var][sel]))
    after = ak.to_numpy(ak.flatten(data_sp[var][sel & keep]))
    plotting.plot_mass_cut_diagnostic(
        before, after, window, hist_def=config['hist_defs'][hist_key],
        label_before='before veto', label_after='after veto',
        title=f'{title} — signal MC', ax=axes[0])

    # --- weighted background MC ---
    b_before, b_after, w_before, w_after = [], [], [], []
    for sp in BACKGROUND_SP_MODES:
        s = baseline_mask & (spmode == sp)
        vb = ak.to_numpy(ak.flatten(data_sp[var][s]))
        va = ak.to_numpy(ak.flatten(data_sp[var][s & keep]))
        b_before.append(vb); b_after.append(va)
        w_before.append(np.full(len(vb), weights[str(sp)]))
        w_after.append(np.full(len(va), weights[str(sp)]))
    plotting.plot_mass_cut_diagnostic(
        np.concatenate(b_before), np.concatenate(b_after), window,
        weights_before=np.concatenate(w_before), weights_after=np.concatenate(w_after),
        hist_def=config['hist_defs'][hist_key],
        label_before='before veto', label_after='after veto',
        title=f'{title} — weighted background MC', ax=axes[1])

    plt.tight_layout()
    return fig

In [ ]:
rd = config['region_definitions']

fig = veto_before_after('BpostFitMes', rd['signal MES'], 'BpostFitMes', r'$m_{ES}$')
fig.savefig(f'{fig_dir}/antibaryon_veto_mes_diagnostic.png', dpi=120, bbox_inches='tight')
plt.show()

fig = veto_before_after('BpostFitDeltaE', rd['signal DeltaE'], 'BpostFitDeltaE', r'$\Delta E$')
fig.savefig(f'{fig_dir}/antibaryon_veto_deltae_diagnostic.png', dpi=120, bbox_inches='tight')
plt.show()

# Null check: the veto should not sculpt the Lambda0 peak.
fig = veto_before_after('Lambda0_unc_Mass',
                        config['composites']['Lambda0']['mass_window'],
                        'Lambda0_unc_Mass', r'$\Lambda^0$ mass (null check)')
fig.savefig(f'{fig_dir}/antibaryon_veto_lambda0_mass_diagnostic.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Cumulative Stage 1–4 performance (Phase 3)

Everything currently in force, in pipeline order, in one table.

**Counting conventions** (easy to misread, so stated explicitly):

- Signal MC and background MC are counted as **$B$ candidates in the
  mES/$\Delta E$ signal region**, matching the Stage 3 Punzi convention.
  Because the signal region is a *subset* of the fit region, the
  "+ fit region" step is a **no-op by construction** for these columns; it is
  kept so the pipeline order is complete.
- Background MC is luminosity-weighted.
- **Collision data is counted in the fit region with the signal box
  REMOVED.** *Blinding:* the signal box is never read. The upstream
  `_BLINDED` files already have it removed, but the mask is applied
  explicitly so the column means what its label says.

In [ ]:
_, data_col = datasets.load_datasets(CHANNEL, sp_or_data='col')
datasets.add_derived_fields(data_col, config)

df_cum = cp.cumulative_cutflow(data_sp, config, weights,
                               data_collision=data_col, veto_selector=recommended)
display(df_cum)

### Per-event surviving-candidate multiplicities

This deliberately uses a **different convention** from the cutflow above: no
single-candidate requirement and no mES/$\Delta E$ region cut, reproducing
Stage 2's and Stage 3's multi-candidate studies. Applying the pipeline's
`nB == 1` cut here would force the multi-candidate fraction to 0 *by
construction* and make the study vacuous — which is precisely the question
the parked single-candidate decision (STATUS.md open decision 1) is asking.

The endpoint is therefore directly comparable to the Stage 2 and Stage 3
numbers. **Numbers only — no candidate-selection policy is applied or
recommended here.**

In [ ]:
mult = cp.candidate_multiplicities(data_sp, config, veto_selector=recommended)

rows = [{'step': e['name'],
         'frac 0 good B': e['frac_zero_good_B'],
         'frac 1 good B': e['frac_one_good_B'],
         'frac >1 good B': e['frac_multi_good_B']} for e in mult.values()]
display(pd.DataFrame(rows))

stored = results_io.load_results(CHANNEL)
for stage in ('stage02', 'stage03'):
    s = stored.get(stage, {}).get('multi_candidate_study', {})
    if s:
        print(f"{stage} recorded in results/{CHANNEL}.yaml: "
              f"{s.get('frac_zero_good_B', float('nan')):.3f} / "
              f"{s.get('frac_one_good_B', float('nan')):.3f} / "
              f"{s.get('frac_multi_good_B', float('nan')):.3f}")
print("\nNOTE: if the stage02 line above disagrees with the 'Stage 2 purity' row,\n"
      "the stored stage02 numbers predate the current channel_config.py and\n"
      "run_stage02.py needs re-running -- see STATUS.md.")

In [ ]:
if CHANNEL == 'Lam0LamC':
    final = mult[max(mult)]
    print("Endpoint candidate multiplicities by role (after the veto):")
    for key in ('n_good_LambdaC_distribution',
                'n_good_Lambda0_from_B_distribution',
                'n_good_Lambda0_from_LambdaC_distribution'):
        print(f"  {key}: {final[key]}")
    print(f"  n_good_B_by_lambdac_mode: {final['n_good_B_by_lambdac_mode']}")

## Observations / checkpoint summary

*(Filled in after executing the notebook for both channels — see the Stage 4
section of the BAD, `results/<channel>.yaml` section `stage04`, and
STATUS.md.)*

**Veto definition.** Per-$B$-candidate, not per-event (the $p\Lambda^0$
reference assumed one $B$ per event; `Lam0LamC` reaches ~60). A candidate is
vetoed if the event holds a track that (a) is not one of *that* candidate's
tracks, (b) passes the chosen KM proton selector, and (c) has charge opposite
to that candidate's own signal baryon charge.

**Structural facts verified empirically, not assumed** (small-sample check,
both channels):

- `pSelectorsMap` is indexed by **track** (`len == nTRK`). The `p` hypothesis
  list is a strict subset of the tracks, and **24% of tracks passing
  `SuperLooseKMProtonSelection` lie outside it** — so the eligible-track pool
  is a real choice. Using the `p` list alone is *degenerate*: it holds only
  ~2 candidates/event (essentially the signal protons, all of signal-baryon
  charge), so the veto almost never fires.
- $\mathrm{sign}(\texttt{TRKLund})$ is the electric charge — it agrees with
  $\mathrm{sign}(\texttt{pLund[pTrkIdx]})$ in 207562 of 207563 proton
  entries (the single disagreement is an isolated ntuple inconsistency).
- The signal baryon sign is consistent within a $B$ candidate: both
  $\Lambda^0$ daughters share a sign (`Lam0Lam0`), and
  $\mathrm{sign}(\texttt{LambdaCLund})$ matches the $\Lambda^0$'s proton
  (`Lam0LamC`), including mode 4's inner $\Lambda^0$ — **0 mismatches** in
  either channel. One uniform rule therefore serves both channels and all
  four modes.

**Exclusion scope.** All of the candidate's tracks are exempt, not just its
proton daughters as in the reference. The signal $\Lambda^0$ pions carry
exactly the antiproton charge, so leaving them eligible lets the signal veto
itself. The two definitions differ on a few % of all candidates; the effect on
*signal* candidates surviving purity + PID inside the box is much smaller.

**Known limitation.** For `Lam0LamC` modes 2 and 3 the $K_S^0$ pion tracks
cannot be resolved from these files and stay eligible to fire the veto (see
§2). Measured cost is small and is recorded per mode.

**Open items for Matt:**

1. Which selector to apply per channel (or whether to adopt the $p\Lambda^0$
   value for consistency across the two analyses).
2. Whether the anti-$\Lambda^0$ add-on is worth its efficiency cost
   (measured: it is not, in either channel).
3. The single-candidate decision (STATUS.md open decision 1) — the endpoint
   multiplicities above are the input, presented not decided.
4. Stale `stage02` multi-candidate numbers in `results/<channel>.yaml`.